# OSC Ink Preparation Log

Registra la preparación de un batch de ink semiconductor (diFT-TES-ADT:PS en chlorobenzene).

**Target:** 2% weight/volume SC + PS in CB, 4:1 ratio SC:PS  
**Rule of thumb:** 22.6 mg reagent per mL CB

In [ ]:
import os, sys
from pathlib import Path
from datetime import datetime

PROJECT_ROOT = Path(os.path.abspath(os.path.join(os.getcwd(), '../..')))
sys.path.insert(0, str(PROJECT_ROOT))

from src import db

db.init_db()
print('Database initialized')

## 1. Weighing

Pesa el SC (diFT-TES-ADT) y PS (10k g/mol) en la balanza.

In [ ]:
# === MODIFICAR ESTOS VALORES ===
SC_WEIGHT_MG = 4.5   # Masa pesada de SC
PS_WEIGHT_MG = 1.1   # Masa pesada de PS
CB_VOLUME_ML = 0.25  # Volumen de chlorobenzene
# ==============================

# Verificar concentración (debe ser ~2% = 22.6 mg/mL)
sc_conc = SC_WEIGHT_MG / CB_VOLUME_ML
ps_conc = PS_WEIGHT_MG / CB_VOLUME_ML
print(f'SC concentration: {sc_conc:.1f} mg/mL (target: 22.6)')
print(f'PS concentration: {ps_conc:.1f} mg/mL (target: 22.6)')
print(f'Actual SC %: {sc_conc / 22.6 * 100:.1f}%')
print(f'Actual PS %: {ps_conc / 22.6 * 100:.1f}%')
print(f'Total volume: {CB_VOLUME_ML * 1000:.0f} µL')
print(f'Estimated devices: ~{int(CB_VOLUME_ML * 1000 / 35)} (at 35 µL/device)')

## 2. Crear memoria de ink batch

In [ ]:
# === MODIFICAR ESTOS VALORES ===
SC_MATERIAL = 'diFT-TES-ADT'
PS_MW = 10000
RATIO = '4:1'
DATE = datetime.now().strftime('%Y-%m-%d')
NOTES = ''
# ==============================

ink_name = datetime.now().strftime('INK-%y%m%d')

memory_id = db.create_memory(
    title=f'Ink Batch {ink_name}',
    type='experiment',
    subtype='session',
    date=DATE,
    description=f'Preparación de ink {SC_MATERIAL} {RATIO} en CB',
    substrate='N/A',
    process='ink preparation',
    custom_fields={
        'sc_material': SC_MATERIAL,
        'sc_weight_mg': str(SC_WEIGHT_MG),
        'ps_weight_mg': str(PS_WEIGHT_MG),
        'ps_mw': str(PS_MW),
        'cb_volume_ml': str(CB_VOLUME_ML),
        'ratio': RATIO,
        'sc_concentration_mg_ml': f'{sc_conc:.1f}',
        'ps_concentration_mg_ml': f'{ps_conc:.1f}',
        'estimated_devices': str(int(CB_VOLUME_ML * 1000 / 35)),
        'notes': NOTES,
    }
)

print(f'Created ink batch memory with id={memory_id}')
print(f'Name: {ink_name}')

## 3. Chlorobenzene Extraction

Extrae CB de la botella bajo atmósfera de Ar:
1. Inflar globo con Ar, insertar aguja pequeña como septum
2. Extraer CB con jeringa + aguja grande (NO inclinar botella)
3. Transferir a vial ámbar

## 4. Dissolution

- Disolver SC y PS por separado en CB (viales ámbar cubiertos con Al foil)
- Colocar en hotplate BAMS a **105°C por ≥ 60 min**
- Mezclar SC:PS en **proporción 4:1**
- Dejar **overnight a 60°C** para uso posterior

## Resumen

In [ ]:
memory = db.get_memory(memory_id)

print(f'=== Ink Batch {memory_id} ===')
print(f'Title: {memory["title"]}')
print(f'Date: {memory["date"]}')

print(f'\nRecipe:')
for f in memory['fields']:
    if f['field_type'] == 'custom':
        print(f'  {f["field_name"]}: {f["field_value"]}')

---
### Calculation Reference

For 2% weight in CB:
- **22.6 mg reagent per mL CB**
- Per device: ~30-40 µL
- For N devices, estimate volume V = N × 35 µL
- Weigh reagent ≈ 22.6 × V(mL) mg, then recalculate with actual weight

Example: 10 devices -> 350 µL -> 22.6 × 0.35 ≈ 7.9 mg SC  
At 4:1, PS needed = SC / 4 ≈ 2.0 mg